# Load IESO Hourly Ontario Energy Price (HOEP) Data

**Purpose:** Load and combine IESO HOEP price data from 2013-2025 (13 yearly files).

**Data Source:** IESO Public Reports - Yearly Hourly HOEP OR Predispatch Report

**Time Range:** 2013-2025 (Note: HOEP retired April 30, 2025)

**Why this matters:** Electricity prices affect demand behavior - high prices may cause industrial users to reduce consumption, creating a two-way relationship between price and demand that improves forecasting accuracy.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Plot settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [2]:
# Load one sample file (2020) to understand the data structure
sample_file = "../../02_Datasets/raw/ieso_prices/PUB_PriceHOEPPredispOR_2020.csv"

# Read the first few rows to see structure
df_sample = pd.read_csv(sample_file)

print(f"Sample file loaded: 2020 HOEP data")
print(f"Shape: {df_sample.shape}")
print(f"\nColumn names:")
print(df_sample.columns.tolist())
print(f"\nFirst 5 rows:")
df_sample.head()

Sample file loaded: 2020 HOEP data
Shape: (8787, 9)

Column names:
['\\\\Yearly HOEP OR Predispatch Report', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8']

First 5 rows:


,\\Yearly HOEP OR Predispatch Report,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,\\Created at 2021-01-31 08:02:40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,\\For 2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Date,Hour,HOEP,Hour 1 Predispatch,Hour 2 Predispatch,Hour 3 Predispatch,OR 10 Min Sync,OR 10 Min non-sync,OR 30 Min
3,2020-01-01,1,0.00,0.00,0.00,0.00,1.03,0.20,0.20
4,2020-01-01,2,0.00,0.00,0.00,0.00,1.04,0.20,0.20


In [3]:
# Load the file again, skipping the first 2 metadata rows
df_sample = pd.read_csv(sample_file, skiprows=2)

print(f"2020 HOEP data loaded correctly")
print(f"Shape: {df_sample.shape}")
print(f"\nColumn names:")
print(df_sample.columns.tolist())
print(f"\nData types:")
print(df_sample.dtypes)
print(f"\nFirst 5 rows:")
df_sample.head()

2020 HOEP data loaded correctly
Shape: (8785, 9)

Column names:
['\\\\For 2020', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8']

Data types:
\\For 2020    object
Unnamed: 1    object
Unnamed: 2    object
Unnamed: 3    object
Unnamed: 4    object
Unnamed: 5    object
Unnamed: 6    object
Unnamed: 7    object
Unnamed: 8    object
dtype: object

First 5 rows:


,\\For 2020,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,Date,Hour,HOEP,Hour 1 Predispatch,Hour 2 Predispatch,Hour 3 Predispatch,OR 10 Min Sync,OR 10 Min non-sync,OR 30 Min
1,2020-01-01,1,0.00,0.00,0.00,0.00,1.03,0.20,0.20
2,2020-01-01,2,0.00,0.00,0.00,0.00,1.04,0.20,0.20
3,2020-01-01,3,0.00,0.00,0.00,0.00,1.05,0.20,0.20
4,2020-01-01,4,0.00,0.00,0.00,-0.02,1.06,0.18,0.18


In [4]:
# Load with proper header row (row 2 = index 2 after skipping metadata)
df_sample = pd.read_csv(sample_file, skiprows=2)

# The first row is still the header labels, let's look at row 0
print("Row 0 (actual headers):")
print(df_sample.iloc[0].values)

# Drop the first row and reset index
df_sample = df_sample.iloc[1:].reset_index(drop=True)

# Rename columns based on what we saw
df_sample.columns = ['Date', 'Hour', 'HOEP', 'Hour_1_Predispatch', 
                     'Hour_2_Predispatch', 'Hour_3_Predispatch',
                     'OR_10Min_Sync', 'OR_10Min_NonSync', 'OR_30Min']

print(f"\n2020 HOEP data cleaned")
print(f"Shape: {df_sample.shape}")
print(f"\nColumns: {df_sample.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(df_sample.head())
print(f"\nData types:")
print(df_sample.dtypes)

Row 0 (actual headers):
['Date' 'Hour' 'HOEP' 'Hour 1 Predispatch' 'Hour 2 Predispatch'
 'Hour 3 Predispatch' 'OR 10 Min Sync' 'OR 10 Min non-sync' 'OR 30 Min']

2020 HOEP data cleaned
Shape: (8784, 9)

Columns: ['Date', 'Hour', 'HOEP', 'Hour_1_Predispatch', 'Hour_2_Predispatch', 'Hour_3_Predispatch', 'OR_10Min_Sync', 'OR_10Min_NonSync', 'OR_30Min']

First 5 rows:
         Date Hour  HOEP Hour_1_Predispatch Hour_2_Predispatch  \
0  2020-01-01    1  0.00               0.00               0.00   
1  2020-01-01    2  0.00               0.00               0.00   
2  2020-01-01    3  0.00               0.00               0.00   
3  2020-01-01    4  0.00               0.00               0.00   
4  2020-01-01    5  0.00               0.00               0.00   

  Hour_3_Predispatch OR_10Min_Sync OR_10Min_NonSync OR_30Min  
0               0.00          1.03             0.20     0.20  
1               0.00          1.04             0.20     0.20  
2               0.00          1.05             

In [5]:
# Convert data types
df_sample['Date'] = pd.to_datetime(df_sample['Date'])
df_sample['Hour'] = pd.to_numeric(df_sample['Hour'])
df_sample['HOEP'] = pd.to_numeric(df_sample['HOEP'], errors='coerce')

# Check date range
print(f"Date range: {df_sample['Date'].min()} to {df_sample['Date'].max()}")
print(f"Total days: {df_sample['Date'].nunique()}")
print(f"Hours per day: {df_sample.groupby('Date')['Hour'].count().value_counts()}")

# Check for missing values
print(f"\nMissing values in HOEP column: {df_sample['HOEP'].isna().sum()}")

# Basic statistics on HOEP
print(f"\nHOEP Statistics ($/MWh):")
print(df_sample['HOEP'].describe())

# Show last few rows
print(f"\nLast 5 rows:")
df_sample.tail()

Date range: 2020-01-01 00:00:00 to 2020-12-31 00:00:00
Total days: 366
Hours per day: Hour
24    366
Name: count, dtype: int64

Missing values in HOEP column: 0

HOEP Statistics ($/MWh):
count    8784.000000
mean       12.651440
std        21.333456
min        -4.510000
25%         3.387500
50%        12.855000
75%        18.340000
max      1258.050000
Name: HOEP, dtype: float64

Last 5 rows:


,Date,Hour,HOEP,Hour_1_Predispatch,Hour_2_Predispatch,Hour_3_Predispatch,OR_10Min_Sync,OR_10Min_NonSync,OR_30Min
8779,2020-12-31,20,19.69,22.43,19.06,21.35,18.89,18.89,0.45
8780,2020-12-31,21,20.78,21.77,21.77,21.37,3.13,3.13,0.30
8781,2020-12-31,22,27.85,20.89,20.91,21.41,7.74,2.83,2.08
8782,2020-12-31,23,19.68,18.65,17.87,21.12,0.77,0.24,0.24
8783,2020-12-31,24,27.37,15.06,14.88,14.88,1.38,0.23,0.23


In [6]:
# Define function to load a single HOEP file
def load_hoep_file(filepath):
    """Load a single HOEP CSV file with proper cleaning."""
    # Read file, skip first 2 metadata rows
    df = pd.read_csv(filepath, skiprows=2)
    
    # Drop first row (header labels) and reset index
    df = df.iloc[1:].reset_index(drop=True)
    
    # Rename columns
    df.columns = ['Date', 'Hour', 'HOEP', 'Hour_1_Predispatch', 
                  'Hour_2_Predispatch', 'Hour_3_Predispatch',
                  'OR_10Min_Sync', 'OR_10Min_NonSync', 'OR_30Min']
    
    # Convert data types
    df['Date'] = pd.to_datetime(df['Date'])
    df['Hour'] = pd.to_numeric(df['Hour'])
    df['HOEP'] = pd.to_numeric(df['HOEP'], errors='coerce')
    
    return df

# Get all HOEP files
price_dir = "../../02_Datasets/raw/ieso_prices/"
price_files = sorted(glob.glob(price_dir + "PUB_PriceHOEPPredispOR_20*.csv"))

print(f"Found {len(price_files)} HOEP files")
print(f"\nLoading all files...")

# Load all files and combine
hoep_dfs = []
for file in price_files:
    year = file.split('_')[-1].replace('.csv', '')
    df = load_hoep_file(file)
    hoep_dfs.append(df)
    print(f"  ✓ Loaded {year}: {len(df):,} rows")

# Combine all dataframes
df_hoep_combined = pd.concat(hoep_dfs, ignore_index=True)

print(f"\n{'='*70}")
print(f"Combined HOEP Dataset:")
print(f"  Total rows: {len(df_hoep_combined):,}")
print(f"  Date range: {df_hoep_combined['Date'].min()} to {df_hoep_combined['Date'].max()}")
print(f"  Missing HOEP values: {df_hoep_combined['HOEP'].isna().sum()}")

Found 13 HOEP files

Loading all files...
  ✓ Loaded 2013: 8,760 rows
  ✓ Loaded 2014: 8,760 rows
  ✓ Loaded 2015: 8,760 rows
  ✓ Loaded 2016: 8,784 rows
  ✓ Loaded 2017: 8,760 rows
  ✓ Loaded 2018: 8,760 rows
  ✓ Loaded 2019: 8,760 rows
  ✓ Loaded 2020: 8,784 rows
  ✓ Loaded 2021: 8,760 rows
  ✓ Loaded 2022: 8,760 rows
  ✓ Loaded 2023: 8,760 rows
  ✓ Loaded 2024: 8,784 rows
  ✓ Loaded 2025: 2,880 rows

Combined HOEP Dataset:
  Total rows: 108,072
  Date range: 2013-01-01 00:00:00 to 2025-04-30 00:00:00
  Missing HOEP values: 5


In [8]:
# Create DateTime column properly handling Hour 24
# Hour 24 means 00:00 of the NEXT day

# First, adjust Hour 24 to Hour 0 of next day
df_hoep_combined['Date_adjusted'] = df_hoep_combined['Date']
df_hoep_combined['Hour_adjusted'] = df_hoep_combined['Hour']

# For Hour 24, change to Hour 0 and add 1 day to date
mask_hour24 = df_hoep_combined['Hour'] == 24
df_hoep_combined.loc[mask_hour24, 'Date_adjusted'] = (
    df_hoep_combined.loc[mask_hour24, 'Date'] + pd.Timedelta(days=1)
)
df_hoep_combined.loc[mask_hour24, 'Hour_adjusted'] = 0

# Now create DateTime (Hour 0-23 are all valid)
df_hoep_combined['DateTime'] = pd.to_datetime(
    df_hoep_combined['Date_adjusted'].astype(str) + ' ' + 
    df_hoep_combined['Hour_adjusted'].astype(str).str.zfill(2) + ':00:00'
)

# Drop temporary columns
df_hoep_combined = df_hoep_combined.drop(['Date_adjusted', 'Hour_adjusted'], axis=1)

# Filter to match your IESO demand timeframe (June 2013 onwards)
df_hoep_filtered = df_hoep_combined[
    df_hoep_combined['DateTime'] >= '2013-06-01'
].copy()

print(f"HOEP data filtered to match IESO demand timeframe:")
print(f"  Date range: {df_hoep_filtered['DateTime'].min()} to {df_hoep_filtered['DateTime'].max()}")
print(f"  Total rows: {len(df_hoep_filtered):,}")
print(f"  Missing HOEP: {df_hoep_filtered['HOEP'].isna().sum()}")

# Show HOEP statistics
print(f"\nHOEP Statistics (2013-2025, $/MWh):")
print(df_hoep_filtered['HOEP'].describe())

# Check first and last few rows
print(f"\nFirst 3 rows:")
print(df_hoep_filtered[['DateTime', 'Date', 'Hour', 'HOEP']].head(3))
print(f"\nLast 3 rows:")
print(df_hoep_filtered[['DateTime', 'Date', 'Hour', 'HOEP']].tail(3))

HOEP data filtered to match IESO demand timeframe:
  Date range: 2013-06-01 00:00:00 to 2025-05-01 00:00:00
  Total rows: 104,449
  Missing HOEP: 5

HOEP Statistics (2013-2025, $/MWh):
count    104444.000000
mean         25.066836
std          32.203873
min        -110.100000
25%           7.720000
50%          20.155000
75%          33.660000
max        1660.800000
Name: HOEP, dtype: float64

First 3 rows:
                DateTime       Date  Hour   HOEP
3623 2013-06-01 00:00:00 2013-05-31    24  13.86
3624 2013-06-01 01:00:00 2013-06-01     1  12.73
3625 2013-06-01 02:00:00 2013-06-01     2  11.21

Last 3 rows:
                  DateTime       Date  Hour   HOEP
108069 2025-04-30 22:00:00 2025-04-30    22  38.12
108070 2025-04-30 23:00:00 2025-04-30    23  36.57
108071 2025-05-01 00:00:00 2025-04-30    24  14.34


In [9]:
# Select only the columns we need for merging
df_hoep_final = df_hoep_filtered[['DateTime', 'HOEP']].copy()

# Sort by DateTime
df_hoep_final = df_hoep_final.sort_values('DateTime').reset_index(drop=True)

# Save to processed folder
output_path = "../../02_Datasets/processed/hoep_prices_2013_2025.csv"
df_hoep_final.to_csv(output_path, index=False)

print(f"✓ HOEP data saved to: {output_path}")
print(f"\nFinal HOEP dataset:")
print(f"  Shape: {df_hoep_final.shape}")
print(f"  Columns: {df_hoep_final.columns.tolist()}")
print(f"  Date range: {df_hoep_final['DateTime'].min()} to {df_hoep_final['DateTime'].max()}")
print(f"  Memory usage: {df_hoep_final.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Preview
print(f"\nPreview:")
df_hoep_final.head(10)

✓ HOEP data saved to: ../../02_Datasets/processed/hoep_prices_2013_2025.csv

Final HOEP dataset:
  Shape: (104449, 2)
  Columns: ['DateTime', 'HOEP']
  Date range: 2013-06-01 00:00:00 to 2025-05-01 00:00:00
  Memory usage: 1.59 MB

Preview:


,DateTime,HOEP
0,2013-06-01 00:00:00,13.86
1,2013-06-01 01:00:00,12.73
2,2013-06-01 02:00:00,11.21
3,2013-06-01 03:00:00,6.65
4,2013-06-01 04:00:00,-3.62
5,2013-06-01 05:00:00,11.40
6,2013-06-01 06:00:00,14.41
7,2013-06-01 07:00:00,13.74
8,2013-06-01 08:00:00,7.47
9,2013-06-01 09:00:00,15.18
